# LCBR vs CL-CBS — Reproducible Multi-Scale Experimental Analysis

## Overview

This notebook provides the complete and reproducible experimental analysis
of **Local Conflict-Based Trajectory Repair (LCBR)** against full-horizon
**Car-Like Conflict-Based Search (CL-CBS)** across the 50×50, 100×100,
and 300×300 benchmark campaigns.

The analysis evaluates LCBR at complementary levels. First, controlled
**same-conflict comparisons** isolate the computational effect of local
trajectory repair by comparing Local and Full SHA* queries for the same
robot, BCT child, and accumulated constraints. Second, **complete solver
analysis** evaluates how these local gains propagate through the actual
conflict-resolution process, including BCT evolution, the number of
low-level calls, cumulative SHA* effort, fallback behavior, and Stage-3
runtime. Finally, the end-to-end runtime decomposition places the
conflict-resolution results within the complete coordination pipeline.

This distinction is essential because a reduction in the cost of an
individual low-level query does not necessarily imply an equivalent
complete-solver speedup. Local and full-horizon replanning may generate
different trajectories and therefore lead to different subsequent
conflicts and low-level workloads. The analysis consequently distinguishes
**per-query efficiency**, **cumulative low-level computational effort**,
**Stage-3 solver performance**, and **end-to-end framework performance**.

The notebook is organized around four experimental data sources:

1. **Normal solver runs** — used to evaluate solver reliability, BCT
   behavior, low-level calls, cumulative SHA* expansions and runtime,
   fallback behavior, Stage-3 performance, and end-to-end runtime.

2. **Paired same-conflict probe runs** — used exclusively for controlled
   Local-vs-Full SHA* comparisons under identical conflict conditions.
   Counterfactual Full queries are diagnostic only and do not influence
   the LCBR search or its measured solver runtime.

3. **Repair-window ablation runs** — used to evaluate the sensitivity of
   LCBR to the repair margin $\delta_w \in \{5,10,20\}$ and characterize
   the trade-off between locality, computational effort, trajectory
   quality, and robustness.

4. **Qualitative full-pipeline scenario** — used to illustrate the complete
   coordination process from robot–POI assignment and nominal trajectory
   generation to collision-free trajectories after LCBR.

All statistical comparisons use explicitly defined experimental
populations, including **both-successful**, **conflict-active**,
**root-only**, and **same-conflict paired** instances where appropriate.
Repeated conflict queries are aggregated at the instance level before
cross-instance statistics are reported.

All reported quantities are computed directly from the experimental logs.
No experimental result is hard-coded as an input. Reference values reported
in the manuscript are used only at the end of the notebook as
reproducibility and consistency checks.

## 1. Setup, campaign paths, and data loading

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, friedmanchisquare

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# ------------------------------------------------------------
# Locate algorithm/ repository root automatically.
# ------------------------------------------------------------
cwd = Path.cwd().resolve()
ROOT = None

for candidate in [cwd, *cwd.parents]:
    if ((candidate / "runs" / "main_comparison").is_dir()
        and (candidate / "experiments" / "main_comparison").is_dir()):
        ROOT = candidate
        break
    nested = candidate / "algorithm"
    if ((nested / "runs" / "main_comparison").is_dir()
        and (nested / "experiments" / "main_comparison").is_dir()):
        ROOT = nested
        break

if ROOT is None:
    raise FileNotFoundError(
        "Could not locate algorithm repository root. "
        "Run this notebook from algorithm/ or a descendant."
    )

RUN_IDS = {
    "50x50": "20260902T204135",
    "100x100": "20260904T014117",
    "300x300": "20260904T100854",
}
RUNS = {s: ROOT / "runs" / "main_comparison" / rid for s, rid in RUN_IDS.items()}

PAPER_EXPORT = ROOT / "runs" / "main_comparison" / "paper_analysis_final"
FIG_EXPORT = PAPER_EXPORT / "figures"
PAPER_EXPORT.mkdir(parents=True, exist_ok=True)
FIG_EXPORT.mkdir(parents=True, exist_ok=True)

def read_scale_csv(scale, filename):
    p = RUNS[scale] / filename
    if not p.is_file():
        raise FileNotFoundError(p)
    d = pd.read_csv(p, low_memory=False)
    d["scale"] = scale
    return d

# Aggregated campaign outputs produced by run_comparison.py
inst = pd.concat([read_scale_csv(s, "instance_metrics.csv") for s in RUNS], ignore_index=True)
queries = pd.concat([read_scale_csv(s, "low_level_queries.csv") for s in RUNS], ignore_index=True)
paired = pd.concat([read_scale_csv(s, "paired_conflicts.csv") for s in RUNS], ignore_index=True)

print("Repository root:", ROOT)
print("instance_metrics:", len(inst))
print("low_level_queries:", len(queries))
print("paired_conflicts:", len(paired))
display(pd.crosstab(inst["scale"], inst["method"], margins=True))

## 2. Experimental populations and integrity checks

In [ ]:
# Stage-3 rows and common-success population
stage3 = inst[inst["status"].ne("PRECHECK_INFEASIBLE")].copy() if "status" in inst else inst.copy()

success_wide = stage3.pivot_table(
    index=["scale","instance_id"],
    columns="method",
    values="success",
    aggfunc="first"
)

common_success_keys = set(
    success_wide[
        success_wide.get("CL-CBS", 0).eq(1) &
        success_wide.get("LCBR", 0).eq(1)
    ].index
)

common_success = stage3[
    stage3.apply(lambda r: (r["scale"], r["instance_id"]) in common_success_keys, axis=1)
].copy()

# Conflict-active means at least one method expanded beyond the root.
bct_wide = common_success.pivot_table(
    index=["scale","instance_id"], columns="method",
    values="bct_nodes_expanded", aggfunc="first"
)
active_mask = (bct_wide["CL-CBS"] > 1) | (bct_wide["LCBR"] > 1)
active_keys = set(bct_wide[active_mask].index)
root_only_keys = set(bct_wide[~active_mask].index)

population = []
for scale in RUNS:
    n_stage3 = stage3.loc[stage3.scale.eq(scale), "instance_id"].nunique()
    good = [k for k in common_success_keys if k[0] == scale]
    active = [k for k in active_keys if k[0] == scale]
    root = [k for k in root_only_keys if k[0] == scale]
    population.append({
        "scale": scale, "stage3_instances": n_stage3,
        "both_successful": len(good), "conflict_active": len(active),
        "root_only": len(root),
        "active_pct_of_both_successful": 100*len(active)/len(good) if good else np.nan,
    })

population = pd.DataFrame(population).set_index("scale")
display(population)
population.to_csv(PAPER_EXPORT/"population_definition.csv")

## 3. Solver reliability and LCBR fallback behavior

In [ ]:
# Reliability categories on paired method availability
s = stage3.pivot_table(
    index=["scale","instance_id"],
    columns="method", values="success", aggfunc="first"
).dropna(subset=["CL-CBS","LCBR"])

def category(r):
    a, b = int(r["CL-CBS"]), int(r["LCBR"])
    if a and b: return "Both"
    if a and not b: return "CL-CBS only"
    if b and not a: return "LCBR only"
    return "Neither"

s["category"] = s.apply(category, axis=1)
reliability = pd.crosstab(s.index.get_level_values("scale"), s["category"])
display(reliability)

lc = stage3[stage3.method.eq("LCBR")].copy()
fallback = lc.groupby("scale").agg(
    local_success=("N_success","sum"),
    local_fail=("N_fail","sum"),
    inapplicable=("N_inapplicable","sum"),
    full_fallback=("N_full_fallback","sum"),
)
fallback["local_decisions"] = fallback[["local_success","local_fail","inapplicable"]].sum(axis=1)
fallback["fallback_rate_pct"] = 100*fallback["full_fallback"]/fallback["local_decisions"]
fallback["applicable_success_rate_pct"] = (
    100*fallback["local_success"] /
    (fallback["local_success"] + fallback["local_fail"]).replace(0,np.nan)
)
display(fallback)

reliability.to_csv(PAPER_EXPORT/"solver_reliability.csv")
fallback.to_csv(PAPER_EXPORT/"fallback_summary.csv")

## 4. Same-conflict Local SHA* vs Full-horizon SHA*

This is the controlled experiment. Each usable local query is paired with a diagnostic full-horizon query for the **same robot, same BCT child, and same accumulated constraints**. The diagnostic full query does not alter the real LCBR search.

In [ ]:
p = paired.copy()
num = [
    "sha_expansions","paired_full_expansions",
    "sha_runtime_s","paired_full_runtime_s",
    "path_length_after","paired_full_path_length_after",
]
for c in num:
    p[c] = pd.to_numeric(p[c], errors="coerce")

valid = (
    p["paired_probe_enabled"].eq(1) &
    p["paired_full_success"].eq(1) &
    p["local_outcome"].ne("inapplicable") &
    p["paired_full_expansions"].gt(0) &
    p["paired_full_runtime_s"].gt(0)
)
q = p.loc[valid].copy()
q["exp_ratio"] = q["sha_expansions"] / q["paired_full_expansions"]
q["runtime_ratio"] = q["sha_runtime_s"] / q["paired_full_runtime_s"]
q["path_ratio"] = q["path_length_after"] / q["paired_full_path_length_after"].replace(0,np.nan)
q["path_penalty"] = q["path_ratio"] - 1

# Experimental unit = instance, not individual conflict.
q_inst = q.groupby(["scale","instance_id"], as_index=False).agg(
    n_pairs=("exp_ratio","count"),
    median_exp_ratio=("exp_ratio","median"),
    median_runtime_ratio=("runtime_ratio","median"),
    median_path_ratio=("path_ratio","median"),
    median_path_penalty=("path_penalty","median"),
)

same_conflict = q_inst.groupby("scale").agg(
    n_instances=("instance_id","nunique"),
    paired_queries=("n_pairs","sum"),
    median_exp_ratio=("median_exp_ratio","median"),
    median_runtime_ratio=("median_runtime_ratio","median"),
    median_path_ratio=("median_path_ratio","median"),
    median_path_penalty=("median_path_penalty","median"),
)
same_conflict["exp_saving_pct"] = 100*(1-same_conflict["median_exp_ratio"])
same_conflict["runtime_saving_pct"] = 100*(1-same_conflict["median_runtime_ratio"])
same_conflict["path_penalty_pct"] = 100*same_conflict["median_path_penalty"]

display(same_conflict)
print("Usable paired queries:", int(q_inst.n_pairs.sum()))
print("Paired instances:", q_inst.instance_id.nunique())

q_inst.to_csv(PAPER_EXPORT/"same_conflict_instance_level.csv", index=False)
same_conflict.to_csv(PAPER_EXPORT/"same_conflict_summary.csv")

### Figure 3 — Same-conflict Local/Full ratios

In [ ]:
scales = ["50x50","100x100","300x300"]
fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))

for ax, col, title in [
    (axes[0], "median_exp_ratio", "(a) SHA* expansions"),
    (axes[1], "median_runtime_ratio", "(b) Low-level runtime"),
]:
    data = [q_inst.loc[q_inst.scale.eq(s), col].dropna().values for s in scales]
    ax.boxplot(data, tick_labels=scales, showfliers=False)
    ax.axhline(1.0, linestyle="--", linewidth=1)
    ax.set_xlabel("Map scale")
    ax.set_title(title)
    for i, vals in enumerate(data, 1):
        med = np.median(vals)
        ax.text(i, med, f"{med:.2f}", ha="center", va="bottom", fontsize=8)

axes[0].set_ylabel("Local / Full ratio")
fig.tight_layout()
fig.savefig(FIG_EXPORT/"fig3_local_vs_full.pdf", bbox_inches="tight")
fig.savefig(FIG_EXPORT/"fig3_local_vs_full.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Repair-window sensitivity ($\delta_w=5,10,20$)

In [ ]:
DW_RUN = ROOT / "runs" / "delta_w_ablation" / "20260905T120004"
DW_VALUES = [5,10,20]

def read_dw(name):
    p = DW_RUN / name
    if not p.is_file(): raise FileNotFoundError(p)
    return pd.read_csv(p, low_memory=False)

dw_inst = read_dw("instance_metrics.csv")
dw_paired = read_dw("paired_conflicts.csv")

dp = dw_paired.copy()
for c in num:
    dp[c] = pd.to_numeric(dp[c], errors="coerce")

valid_dw = (
    dp["paired_probe_enabled"].eq(1) &
    dp["paired_full_success"].eq(1) &
    dp["sha_expansions"].gt(0) &
    dp["paired_full_expansions"].gt(0) &
    dp["paired_full_runtime_s"].gt(0)
)
pv = dp.loc[valid_dw].copy()
pv["exp_ratio"] = pv["sha_expansions"]/pv["paired_full_expansions"]
pv["runtime_ratio"] = pv["sha_runtime_s"]/pv["paired_full_runtime_s"]
pv["path_ratio"] = pv["path_length_after"]/pv["paired_full_path_length_after"].replace(0,np.nan)
pv["path_penalty"] = pv["path_ratio"]-1

dw_pair_inst = pv.groupby(["instance_id","delta_w_steps"], as_index=False).agg(
    n_pairs=("exp_ratio","count"),
    median_exp_ratio=("exp_ratio","median"),
    median_runtime_ratio=("runtime_ratio","median"),
    median_path_ratio=("path_ratio","median"),
    median_path_penalty=("path_penalty","median"),
)

window_sensitivity = dw_pair_inst.groupby("delta_w_steps").agg(
    n_instances=("instance_id","nunique"),
    paired_queries=("n_pairs","sum"),
    median_exp_ratio=("median_exp_ratio","median"),
    median_runtime_ratio=("median_runtime_ratio","median"),
    median_path_penalty=("median_path_penalty","median"),
).reindex(DW_VALUES)

window_sensitivity["exp_saving_pct"] = 100*(1-window_sensitivity["median_exp_ratio"])
window_sensitivity["runtime_saving_pct"] = 100*(1-window_sensitivity["median_runtime_ratio"])
window_sensitivity["path_penalty_pct"] = 100*window_sensitivity["median_path_penalty"]

robustness = dw_inst.groupby("delta_w_steps").agg(
    n_instances=("instance_id","nunique"),
    successes=("success","sum"),
    timeouts=("timeout","sum"),
).reindex(DW_VALUES)

display(window_sensitivity)
display(robustness)
window_sensitivity.to_csv(PAPER_EXPORT/"delta_w_sensitivity.csv")
robustness.to_csv(PAPER_EXPORT/"delta_w_robustness.csv")

### Figure 4 — Repair-window sensitivity

In [ ]:
dw = np.array(DW_VALUES)
fig, ax = plt.subplots(figsize=(3.5,2.7))
ax.plot(dw, window_sensitivity.loc[dw,"median_exp_ratio"], marker="o", label="SHA* expansions")
ax.plot(dw, window_sensitivity.loc[dw,"median_runtime_ratio"], marker="s", label="Query runtime")
ax.axhline(1.0, linestyle="--", linewidth=1)
ax.set_xticks(dw)
ax.set_xlabel(r"Repair margin $\delta_w$ [steps]")
ax.set_ylabel("Local / Full ratio")
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIG_EXPORT/"fig4_delta_w_sensitivity.pdf", bbox_inches="tight")
fig.savefig(FIG_EXPORT/"fig4_delta_w_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. From Per-Query Efficiency to Solver-Level Performance

The same-conflict experiment quantifies the intrinsic computational benefit
of restricting a SHA* replanning query to a local trajectory window. However,
this controlled per-query comparison does not directly measure the performance
of the complete conflict-resolution process.

Once CL-CBS and LCBR evolve independently, their replanning decisions may
produce different trajectories, which can lead to different subsequent
conflicts, BCT evolution, numbers of low-level calls, and cumulative SHA*
work. Consequently, a large reduction in the cost of an individual Local
SHA* query does not necessarily translate into an equivalent reduction in
complete Stage-3 runtime.

This section therefore analyzes the transition from **per-query efficiency**
to **solver-level performance** by examining:

- BCT nodes expanded;
- number of actual low-level SHA* calls;
- cumulative SHA* expansions;
- cumulative SHA* runtime;
- LCBR fallback behavior;
- Stage-3 runtime; and
- the fraction of instances in which conflict repair is actually activated.

Only **normal solver runs** from `_scratch_logs` are used for this analysis.
These runs have `paired_probe_calls = 0`; therefore, the counterfactual
full-horizon queries used in the same-conflict experiment do not contribute
to the measured solver runtime.

### Analysis populations

To avoid mixing fundamentally different solver situations, the following
populations are distinguished:

- **Both-successful:** instances successfully solved by both CL-CBS and LCBR.
  These form the primary population for direct solver-level comparisons.

- **Conflict-active:** both-successful instances for which at least one method
  expands beyond the BCT root (`bct_nodes_expanded > 1`). These instances
  require actual conflict-resolution activity and therefore provide the
  relevant population for studying how local-repair savings propagate through
  the solver.

- **Root-only:** both-successful instances for which both methods terminate
  after expanding only the BCT root. No conflict-triggered low-level
  replanning is required in these cases, so LCBR has no local repair operation
  to accelerate.

- **Real low-level calls:** actual SHA* calls executed during normal solver
  operation. For CL-CBS these correspond to full-horizon replanning queries;
  for LCBR they include local repair queries and any full-horizon fallback
  queries required when local repair is infeasible or inapplicable.

This separation allows the analysis to determine where the computational
advantage observed in controlled same-conflict queries is attenuated when
LCBR is embedded in the complete CL-CBS search process.

In [ ]:
# Load raw normal logs used for real solver behavior.
normal_inst_parts, normal_q_parts = [], []

for scale, run in RUNS.items():
    base = run / "_scratch_logs"
    fi, fq = base/"instance_log.csv", base/"query_log.csv"
    if not fi.is_file() or not fq.is_file():
        raise FileNotFoundError(f"Missing normal scratch logs for {scale}: {base}")
    a, b = pd.read_csv(fi), pd.read_csv(fq)
    a["scale"], b["scale"] = scale, scale
    normal_inst_parts.append(a); normal_q_parts.append(b)

ni = pd.concat(normal_inst_parts, ignore_index=True)
nq = pd.concat(normal_q_parts, ignore_index=True)

# Guard against accidental probe contamination.
assert ni["paired_probe_calls"].fillna(0).sum() == 0
assert ni["paired_probe_runtime_s"].fillna(0).sum() == 0

# Both-successful population.
sw = ni.pivot_table(index=["scale","instance_id"], columns="method", values="success", aggfunc="first").dropna()
good_idx = sw[(sw["CL-CBS"]==1)&(sw["LCBR"]==1)].index
good = set(good_idx)

ni_good = ni[ni.apply(lambda r:(r.scale,r.instance_id) in good, axis=1)].copy()
nq_good = nq[nq.apply(lambda r:(r.scale,r.instance_id) in good, axis=1)].copy()

bw = ni_good.pivot_table(index=["scale","instance_id"], columns="method", values="bct_nodes_expanded", aggfunc="first")
active_idx = bw[(bw["CL-CBS"]>1)|(bw["LCBR"]>1)].index
active = set(active_idx)
root_idx = bw[(bw["CL-CBS"]==1)&(bw["LCBR"]==1)].index

pop_rows=[]
for scale in RUNS:
    g=[x for x in good if x[0]==scale]
    a=[x for x in active if x[0]==scale]
    r=[x for x in root_idx if x[0]==scale]
    pop_rows.append([scale,len(g),len(a),len(r),100*len(a)/len(g)])
solver_population=pd.DataFrame(pop_rows,columns=["scale","both_successful","conflict_active","root_only","active_pct"])
display(solver_population)

### 6.1 BCT nodes, low-level calls, cumulative SHA* work, and Stage-3 runtime

In [ ]:
# Real low-level effort per instance/method.
ll = nq_good.groupby(["scale","instance_id","method"]).agg(
    N_LL=("sha_runtime_s","size"),
    T_LL=("sha_runtime_s","sum"),
    E_LL=("sha_expansions","sum"),
).reset_index()

# Add zeros for both-successful instances with no low-level call.
grid = pd.DataFrame(
    [(s,i,m) for (s,i) in good for m in ["CL-CBS","LCBR"]],
    columns=["scale","instance_id","method"]
)
ll = grid.merge(ll, on=["scale","instance_id","method"], how="left").fillna(
    {"N_LL":0,"T_LL":0.0,"E_LL":0}
)

def paired_median_ratio(df, value, ids):
    x = df[df.apply(lambda r:(r.scale,r.instance_id) in ids, axis=1)]
    p = x.pivot_table(index=["scale","instance_id"], columns="method", values=value, aggfunc="first")
    out=[]
    for scale in RUNS:
        z=p.loc[scale] if scale in p.index.get_level_values(0) else pd.DataFrame()
        if len(z)==0: continue
        valid=z["CL-CBS"].gt(0)&z["LCBR"].notna()
        rr=z.loc[valid,"LCBR"]/z.loc[valid,"CL-CBS"]
        out.append([scale,len(rr),rr.median(),rr.quantile(.25),rr.quantile(.75)])
    return pd.DataFrame(out,columns=["scale","n","median_ratio","q1","q3"]).set_index("scale")

# BCT ratios on conflict-active population.
bct_long = ni_good[["scale","instance_id","method","bct_nodes_expanded"]]
bct_ratio = paired_median_ratio(bct_long,"bct_nodes_expanded",active)
nll_ratio = paired_median_ratio(ll,"N_LL",active)
tll_ratio = paired_median_ratio(ll,"T_LL",active)
ell_ratio = paired_median_ratio(ll,"E_LL",active)

stage3_long = ni_good[["scale","instance_id","method","conflict_resolution_runtime_s"]]
stage3_active_ratio = paired_median_ratio(stage3_long,"conflict_resolution_runtime_s",active)

attenuation = pd.DataFrame(index=RUNS.keys())
attenuation["BCT_ratio_active"] = bct_ratio["median_ratio"]
attenuation["LL_calls_ratio_active"] = nll_ratio["median_ratio"]
attenuation["cum_LL_runtime_ratio_active"] = tll_ratio["median_ratio"]
attenuation["cum_LL_runtime_saving_pct"] = 100*(1-attenuation["cum_LL_runtime_ratio_active"])
attenuation["cum_LL_exp_ratio_active"] = ell_ratio["median_ratio"]
attenuation["cum_LL_exp_saving_pct"] = 100*(1-attenuation["cum_LL_exp_ratio_active"])
attenuation["stage3_ratio_active"] = stage3_active_ratio["median_ratio"]
attenuation["stage3_saving_active_pct"] = 100*(1-attenuation["stage3_ratio_active"])

# Overall Stage-3 paired ratio on all both-successful instances.
stage3_all_ratio = paired_median_ratio(stage3_long,"conflict_resolution_runtime_s",good)
attenuation["stage3_ratio_all_successful"] = stage3_all_ratio["median_ratio"]
attenuation["stage3_saving_all_successful_pct"] = 100*(1-attenuation["stage3_ratio_all_successful"])

display(attenuation)
attenuation.to_csv(PAPER_EXPORT/"solver_attenuation_summary.csv")

### 6.2 BCT and low-level summary (conflict-active instances)

In [ ]:
def fmt_iqr(x):
    x=pd.Series(x).dropna()
    return f"{x.median():.3f} [{x.quantile(.25):.3f}-{x.quantile(.75):.3f}]"

ll_active = ll[ll.apply(lambda r:(r.scale,r.instance_id) in active, axis=1)]
z = ni_good[ni_good.apply(lambda r:(r.scale,r.instance_id) in active, axis=1)].merge(
    ll_active, on=["scale","instance_id","method"], how="left"
)
z["sha_share_stage3_pct"] = np.where(
    z["conflict_resolution_runtime_s"]>0,
    100*z["T_LL"]/z["conflict_resolution_runtime_s"], np.nan
)

rows=[]
for scale in RUNS:
    for method in ["CL-CBS","LCBR"]:
        x=z[(z.scale==scale)&(z.method==method)]
        rows.append({
            "Scale":scale, "Method":method, "N active":len(x),
            "BCT nodes median [IQR]":fmt_iqr(x.bct_nodes_expanded),
            "LL calls median [IQR]":fmt_iqr(x.N_LL),
            "Total LL expansions median [IQR]":fmt_iqr(x.E_LL),
            "Total LL runtime s median [IQR]":fmt_iqr(x.T_LL),
            "Stage3 runtime s median [IQR]":fmt_iqr(x.conflict_resolution_runtime_s),
            "SHA* share Stage3 % median [IQR]":fmt_iqr(x.sha_share_stage3_pct),
        })
active_summary=pd.DataFrame(rows)
display(active_summary)
active_summary.to_csv(PAPER_EXPORT/"conflict_active_publication_summary.csv",index=False)

### 6.3 Runtime breakdown

In [ ]:
# Add cumulative real SHA* time to both-successful instance records.
sha = nq_good.groupby(["scale","instance_id","method"])["sha_runtime_s"].sum().rename("sha_total_s").reset_index()
rb = ni_good.merge(sha,on=["scale","instance_id","method"],how="left")
rb["sha_total_s"]=rb["sha_total_s"].fillna(0.0)
rb["stage3_non_sha_s"]=rb["conflict_resolution_runtime_s"]-rb["sha_total_s"]
rb["nominal_share_pct"]=100*rb["nominal_gen_runtime_s"]/rb["total_runtime_s"]
rb["stage3_share_pct"]=100*rb["conflict_resolution_runtime_s"]/rb["total_runtime_s"]
rb["sha_stage3_share_pct"]=np.where(
    rb["conflict_resolution_runtime_s"]>0,
    100*rb["sha_total_s"]/rb["conflict_resolution_runtime_s"],np.nan
)

rows=[]
for population_name, ids in [("all_successful",good),("conflict_active",active)]:
    xx=rb[rb.apply(lambda r:(r.scale,r.instance_id) in ids,axis=1)]
    for scale in RUNS:
        for method in ["CL-CBS","LCBR"]:
            x=xx[(xx.scale==scale)&(xx.method==method)]
            rows.append({
                "scale":scale,"population":population_name,"method":method,"n":len(x),
                "median_total_s":x.total_runtime_s.median(),
                "median_nominal_s":x.nominal_gen_runtime_s.median(),
                "median_lap_s":x.lap_runtime_s.median(),
                "median_stage3_s":x.conflict_resolution_runtime_s.median(),
                "median_sha_total_s":x.sha_total_s.median(),
                "median_stage3_non_sha_s":x.stage3_non_sha_s.median(),
                "median_nominal_share_pct":x.nominal_share_pct.median(),
                "median_stage3_share_pct":x.stage3_share_pct.median(),
                "median_sha_stage3_share_pct":x.sha_stage3_share_pct.median(),
            })

runtime_breakdown=pd.DataFrame(rows)
display(runtime_breakdown)
runtime_breakdown.to_csv(PAPER_EXPORT/"runtime_breakdown.csv",index=False)

### 6.4 Fallback contribution

In [ ]:
fallback_normal = ni[ni.method.eq("LCBR")].groupby("scale").agg(
    local_success=("N_success","sum"),
    local_fail=("N_fail","sum"),
    inapplicable=("N_inapplicable","sum"),
    full_fallback=("N_full_fallback","sum"),
)
fallback_normal["local_decisions"]=fallback_normal[["local_success","local_fail","inapplicable"]].sum(axis=1)
fallback_normal["fallback_rate_pct"]=100*fallback_normal.full_fallback/fallback_normal.local_decisions
display(fallback_normal)
fallback_normal.to_csv(PAPER_EXPORT/"normal_solver_fallback_summary.csv")

### 6.5 Interpretation supported by the logs

The analysis separates three effects rather than treating the 76% low-level gain as an end-to-end speedup:

- **Per-query effect:** the same-conflict probe isolates the cost reduction obtained by shortening one SHA* query.
- **Cumulative solver effect:** once CL-CBS and LCBR evolve independently, they can execute different numbers/sequences of low-level queries. The cumulative SHA* runtime and expansions therefore show much smaller savings than the controlled per-query comparison.
- **Activation effect:** many successful large-map instances terminate at the BCT root, so no conflict repair is invoked and LCBR has no low-level work to accelerate.

The runtime breakdown additionally tests whether non-SHA* BCT overhead explains the attenuation. On conflict-active instances, the logged SHA* calls account for almost all Stage-3 runtime, so the principal attenuation occurs in the transition from **per-query efficiency** to **cumulative low-level workload**, not in a dominant high-level overhead.

## 7. Compact attenuation figure

### Reading the 300×300 attenuation result

The same-conflict experiment isolates the computational effect of
restricting a single SHA* query to a local repair window. On 300×300
maps, this reduces median query runtime by 76.42%.

This value is not a complete-solver speedup. When CL-CBS and LCBR
evolve independently on conflict-active instances, LCBR executes a
different sequence of low-level queries. The median BCT-node ratio
remains 1.0, whereas the median number of low-level calls increases
from 2 for CL-CBS to 3 for LCBR.

Consequently, the cumulative low-level runtime saving becomes 3.47%,
closely matching the 3.27% Stage-3 saving on conflict-active
instances. Furthermore, 105 of the 146 commonly solved 300×300
instances terminate at the BCT root, where no conflict repair is
required. Including these instances reduces the median Stage-3
saving to 0.46%.

The logged SHA* calls account for approximately 99% of Stage-3
runtime on conflict-active 300×300 instances. Therefore, the
attenuation is primarily associated with the transition from
per-query efficiency to cumulative low-level workload rather than
with dominant high-level BCT overhead.

In [ ]:
# This is an explanatory analysis figure, not one of the original manuscript figures.
# It visualizes the three levels without mixing their statistical populations.
scale="300x300"
per_query = 100*(1-same_conflict.loc[scale,"median_runtime_ratio"])
cum_ll = attenuation.loc[scale,"cum_LL_runtime_saving_pct"]
active_s3 = attenuation.loc[scale,"stage3_saving_active_pct"]
all_s3 = attenuation.loc[scale,"stage3_saving_all_successful_pct"]

labels=["Same-conflict\nquery","Cumulative LL\n(active)","Stage 3\n(active)","Stage 3\n(all successful)"]
vals=[per_query,cum_ll,active_s3,all_s3]

fig,ax=plt.subplots(figsize=(5.5,3.0))
ax.bar(labels,vals)
ax.set_ylabel("Median runtime saving [%]")
ax.set_title("300×300: attenuation from per-query to solver level")
for i,v in enumerate(vals):
    ax.text(i,v,f"{v:.2f}%",ha="center",va="bottom",fontsize=8)
fig.tight_layout()
fig.savefig(FIG_EXPORT/"solver_attenuation_300x300.pdf",bbox_inches="tight")
fig.savefig(FIG_EXPORT/"solver_attenuation_300x300.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 3.2))

ax.bar(labels, vals)

ax.set_ylabel("Median runtime saving [%]")
ax.set_title(
    "300×300: attenuation from per-query to solver level"
)

for i, v in enumerate(vals):
    ax.text(
        i, v,
        f"{v:.2f}%",
        ha="center",
        va="bottom",
        fontsize=8
    )

# BCT / LL-call diagnostic
ax.text(
    0.98, 0.95,
    "Conflict-active instances:\n"
    "Median BCT nodes: 2 (CL-CBS) vs 2 (LCBR)\n"
    "Median low-level calls: 2 vs 3",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=8,
    bbox=dict(
        boxstyle="round",
        facecolor="white",
        alpha=0.85
    )
)

fig.tight_layout()

fig.savefig(
    FIG_EXPORT / "solver_attenuation_300x300.pdf",
    bbox_inches="tight"
)

fig.savefig(
    FIG_EXPORT / "solver_attenuation_300x300.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 8. Figure 2 — Qualitative end-to-end framework example

In [ ]:
# ============================================================
# QUALITATIVE FULL-FRAMEWORK PATHS
# ============================================================

FULL_PIPELINE_ROOT = ROOT / "runs" / "full_pipeline"

RUN_DIR = FULL_PIPELINE_ROOT / "pineapple_6ugv"

MAP_FILE = (
    ROOT
    / "experiments"
    / "full_pipeline"
    / "instances"
    / "paper_pineapple_field_6ugv.yaml"
)

NOMINAL_FILE = RUN_DIR / "nominal.yaml"
FINAL_FILE = RUN_DIR / "final_lcbr.yaml"
ASSIGNMENT_FILE = RUN_DIR / "assignment.csv"
COST_MATRIX_FILE = RUN_DIR / "cost_matrix.csv"

FIG_DIR = RUN_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Validation
# ============================================================

FILES = {
    "scenario": MAP_FILE,
    "nominal": NOMINAL_FILE,
    "final": FINAL_FILE,
    "assignment": ASSIGNMENT_FILE,
    "cost matrix": COST_MATRIX_FILE,
}

print("=" * 80)
print("FULL-FRAMEWORK QUALITATIVE DATA")
print("=" * 80)

for name, path in FILES.items():
    print(f"{name:12s}: {path}")
    print(f"{'':12s}  -> {'OK' if path.is_file() else 'MISSING'}")

missing = [
    str(path)
    for path in FILES.values()
    if not path.is_file()
]

if missing:
    raise FileNotFoundError(
        "\nMissing qualitative files:\n  "
        + "\n  ".join(missing)
    )

print()
print("All qualitative files found.")

In [ ]:
# ============================================================
# LOAD QUALITATIVE END-TO-END DATA
# ============================================================

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def load_yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)


scenario = load_yaml(MAP_FILE)
nominal = load_yaml(NOMINAL_FILE)
final = load_yaml(FINAL_FILE)

assignment = pd.read_csv(ASSIGNMENT_FILE)
cost_matrix = pd.read_csv(COST_MATRIX_FILE)

robots = scenario["robots"]
pois = scenario["pois"]
dimensions = scenario["map"]["dimensions"]

obstacles = np.asarray(
    scenario["map"]["obstacles"],
    dtype=float
)


def extract_schedule(solution):
    trajectories = {}

    for key, states in solution["schedule"].items():

        robot_id = int(str(key).replace("agent", ""))

        trajectories[robot_id] = np.asarray(
            [[float(s["x"]), float(s["y"])] for s in states],
            dtype=float
        )

    return trajectories


gamma0 = extract_schedule(nominal)
gamma_star = extract_schedule(final)


# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

M = len(robots)

assert len(pois) == M
assert len(gamma0) == M
assert len(gamma_star) == M
assert len(assignment) == M

print("=" * 70)
print("QUALITATIVE END-TO-END SCENARIO")
print("=" * 70)

print("Robots             :", M)
print("POIs               :", len(pois))
print("Nominal trajectories:", len(gamma0))
print("Final trajectories  :", len(gamma_star))
print("Obstacles           :", len(obstacles))

print("\nAssignment:")
display(assignment)

print("\nNominal statistics:")
print(nominal.get("statistics", {}))

print("\nFinal statistics:")
print(final.get("statistics", {}))

In [ ]:
# ============================================================
# THREE-STAGE END-TO-END FRAMEWORK FIGURE
# ============================================================

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]


def setup_axis(ax):
    """Draw agricultural environment."""

    ax.scatter(
        obstacles[:, 0],
        obstacles[:, 1],
        marker="s",
        s=20,
        color="black",
        zorder=1
    )

    ax.set_xlim(0, dimensions[0])
    ax.set_ylim(0, dimensions[1])

    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel("x [m]")
    ax.grid(False)


def draw_robots_and_pois(ax):
    """Draw initial robot configurations and POIs."""

    # Robots
    for i, robot in enumerate(robots):

        x, y, yaw = robot["start"]
        c = colors[i % len(colors)]

        ax.scatter(
            x, y,
            s=48,
            color=c,
            edgecolor="black",
            linewidth=0.6,
            zorder=6
        )

        ax.annotate(
            rf"$R_{{{i+1}}}$",
            (x, y),
            xytext=(0, -10),
            textcoords="offset points",
            ha="center",
            va="top",
            fontsize=8
        )

    # POIs
    for j, poi in enumerate(pois):

        x, y, yaw = poi["goal"]

        ax.scatter(
            x, y,
            marker="*",
            s=90,
            color="red",
            edgecolor="black",
            linewidth=0.5,
            zorder=7
        )

        ax.annotate(
            rf"$P_{{{j+1}}}$",
            (x, y),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8
        )


def draw_assignment(ax):
    """Robot-to-POI assignment obtained by Hungarian."""

    for _, row in assignment.iterrows():

        i = int(row["robot_id"])
        j = int(row["poi_id"])

        sx, sy, _ = robots[i]["start"]
        gx, gy, _ = pois[j]["goal"]

        ax.plot(
            [sx, gx],
            [sy, gy],
            linestyle=":",
            linewidth=0.8,
            color=colors[i % len(colors)],
            alpha=0.45,
            zorder=2
        )


def draw_paths(ax, trajectories, linestyle="-"):

    for i, xy in sorted(trajectories.items()):

        ax.plot(
            xy[:, 0],
            xy[:, 1],
            linestyle=linestyle,
            linewidth=1.6,
            color=colors[i % len(colors)],
            zorder=4
        )


# ============================================================
# Figure
# ============================================================

fig, axes = plt.subplots(
    1, 3,
    figsize=(12.0, 4.1),
    sharex=True,
    sharey=True
)


# ------------------------------------------------------------
# (a) Initial scenario + resulting assignment
# ------------------------------------------------------------

setup_axis(axes[0])
draw_robots_and_pois(axes[0])
draw_assignment(axes[0])

axes[0].set_ylabel("y [m]")
axes[0].set_title("(a) Initial scenario", fontsize=10)


# ------------------------------------------------------------
# (b) Stage 1 + Stage 2
# ------------------------------------------------------------

setup_axis(axes[1])
draw_paths(
    axes[1],
    gamma0,
    linestyle="--"
)
draw_robots_and_pois(axes[1])

axes[1].set_title(
    r"(b) After assignment: $\Gamma^0$",
    fontsize=10
)


# ------------------------------------------------------------
# (c) Stage 3 -- LCBR
# ------------------------------------------------------------

setup_axis(axes[2])
draw_paths(
    axes[2],
    gamma_star,
    linestyle="-"
)
draw_robots_and_pois(axes[2])

axes[2].set_title(
    r"(c) After LCBR: $\Gamma^\star$",
    fontsize=10
)


plt.tight_layout()


# ============================================================
# Export
# ============================================================

PNG_FILE = FIG_DIR / "full_framework_three_stage.png"
PDF_FILE = FIG_DIR / "full_framework_three_stage.pdf"

fig.savefig(
    PNG_FILE,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    PDF_FILE,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(PNG_FILE)
print(PDF_FILE)

fig.savefig(FIG_EXPORT / "fig2_full_framework_three_stage.pdf", bbox_inches="tight")
fig.savefig(FIG_EXPORT / "fig2_full_framework_three_stage.png", dpi=300, bbox_inches="tight")

## 9. Manuscript consistency checks

In [ ]:
# These values are verification targets taken from the current manuscript.
# They are NOT used to generate any result above.
def close(a,b,tol=0.01):
    return np.isfinite(a) and abs(a-b)<=tol

checks = []

# Same-conflict manuscript ratios.
targets = {
    "50x50": (1.000,1.012),
    "100x100":(0.820,0.843),
    "300x300":(0.252,0.236),
}
for s,(er,rr) in targets.items():
    checks += [
        (f"{s} expansion ratio", same_conflict.loc[s,"median_exp_ratio"], er, close(same_conflict.loc[s,"median_exp_ratio"],er,0.01)),
        (f"{s} runtime ratio", same_conflict.loc[s,"median_runtime_ratio"], rr, close(same_conflict.loc[s,"median_runtime_ratio"],rr,0.01)),
    ]

# delta_w manuscript ratios.
dw_targets={5:(0.146,0.1668),10:(0.6014,0.6113),20:(1.0063,1.0315)}
for w,(er,rr) in dw_targets.items():
    checks += [
        (f"dw={w} expansion ratio",window_sensitivity.loc[w,"median_exp_ratio"],er,close(window_sensitivity.loc[w,"median_exp_ratio"],er,0.015)),
        (f"dw={w} runtime ratio",window_sensitivity.loc[w,"median_runtime_ratio"],rr,close(window_sensitivity.loc[w,"median_runtime_ratio"],rr,0.02)),
    ]


# ============================================================
# Reliability / fallback manuscript checks
# ============================================================

total_local_success = fallback["local_success"].sum()
total_local_fail = fallback["local_fail"].sum()

overall_applicable_success = (
    100.0
    * total_local_success
    / (total_local_success + total_local_fail)
)

total_fallback = fallback["full_fallback"].sum()
total_decisions = fallback["local_decisions"].sum()

overall_fallback_rate = (
    100.0
    * total_fallback
    / total_decisions
)

fallback_300 = fallback.loc[
    "300x300",
    "fallback_rate_pct"
]


checks += [

    (
        "Applicable local-repair success [%]",
        overall_applicable_success,
        99.66,
        close(
            overall_applicable_success,
            99.66,
            0.02
        )
    ),

    (
        "Overall fallback rate [%]",
        overall_fallback_rate,
        20.31,
        close(
            overall_fallback_rate,
            20.31,
            0.02
        )
    ),

    (
        "300x300 fallback rate [%]",
        fallback_300,
        6.25,
        close(
            fallback_300,
            6.25,
            0.01
        )
    ),
]
check_df=pd.DataFrame(checks,columns=["metric","computed","manuscript_target","pass"])
display(check_df)
check_df.to_csv(PAPER_EXPORT/"manuscript_consistency_checks.csv",index=False)

if not check_df["pass"].all():
    print("WARNING: at least one manuscript consistency check failed. Inspect before reporting.")
else:
    print("All manuscript consistency checks passed.")

## 9.1 — Final paper and supervisor summary

This section consolidates the manuscript results and the additional
solver-level diagnostics used to explain why the large same-conflict
low-level gains translate only partially to complete Stage-3 runtime.

All quantities below are computed from the experimental logs.
No result is manually entered.

In [ ]:
# ============================================================
# FINAL PAPER + SUPERVISOR SUMMARY
# ============================================================

summary_rows = []

for scale in ["50x50", "100x100", "300x300"]:

    # --------------------------------------------------------
    # Same-conflict results
    # --------------------------------------------------------

    sc = same_conflict.loc[scale]

    # --------------------------------------------------------
    # Solver attenuation
    # --------------------------------------------------------

    att = attenuation.loc[scale]

    # --------------------------------------------------------
    # Population
    # --------------------------------------------------------

    pop = solver_population[
        solver_population["scale"] == scale
    ].iloc[0]

    # --------------------------------------------------------
    # Fallback
    # --------------------------------------------------------

    fb = fallback_normal.loc[scale]

    summary_rows.append({

        "Scale": scale,

        # Population
        "Both successful":
            int(pop["both_successful"]),

        "Conflict-active":
            int(pop["conflict_active"]),

        "Root-only":
            int(pop["root_only"]),

        "Active [%]":
            pop["active_pct"],

        # Same-conflict
        "Same-conflict exp. saving [%]":
            sc["exp_saving_pct"],

        "Same-conflict runtime saving [%]":
            sc["runtime_saving_pct"],

        "Path penalty [%]":
            sc["path_penalty_pct"],

        # Real solver
        "BCT ratio (active)":
            att["BCT_ratio_active"],

        "LL-call ratio (active)":
            att["LL_calls_ratio_active"],

        "Cumulative LL exp. saving [%]":
            att["cum_LL_exp_saving_pct"],

        "Cumulative LL runtime saving [%]":
            att["cum_LL_runtime_saving_pct"],

        "Stage-3 saving active [%]":
            att["stage3_saving_active_pct"],

        "Stage-3 saving all successful [%]":
            att["stage3_saving_all_successful_pct"],

        # Fallback
        "Fallback [%]":
            fb["fallback_rate_pct"],
    })


final_summary = pd.DataFrame(summary_rows)


# ============================================================
# Display
# ============================================================

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.2f}"
)

print("=" * 110)
print("FINAL PAPER + SUPERVISOR SUMMARY")
print("=" * 110)

display(final_summary)


# ============================================================
# Export
# ============================================================

final_summary.to_csv(
    PAPER_EXPORT / "FINAL_paper_supervisor_summary.csv",
    index=False
)

print(
    "\nSaved:",
    PAPER_EXPORT / "FINAL_paper_supervisor_summary.csv"
)

In [ ]:
# ============================================================
# SUPERVISOR-QUESTION CONSISTENCY CHECK
# ============================================================

s = "300x300"

assert np.isclose(
    same_conflict.loc[s, "runtime_saving_pct"],
    76.424811,
    atol=0.02
)

assert np.isclose(
    attenuation.loc[s, "BCT_ratio_active"],
    1.0,
    atol=0.01
)

assert np.isclose(
    attenuation.loc[s, "LL_calls_ratio_active"],
    1.5,
    atol=0.01
)

assert np.isclose(
    attenuation.loc[s, "cum_LL_runtime_saving_pct"],
    3.465914,
    atol=0.02
)

assert np.isclose(
    attenuation.loc[s, "cum_LL_exp_saving_pct"],
    3.187522,
    atol=0.02
)

assert np.isclose(
    attenuation.loc[s, "stage3_saving_active_pct"],
    3.266417,
    atol=0.02
)

assert np.isclose(
    attenuation.loc[s, "stage3_saving_all_successful_pct"],
    0.461983,
    atol=0.02
)

assert int(
    solver_population.loc[
        solver_population["scale"] == s,
        "conflict_active"
    ].iloc[0]
) == 41

assert int(
    solver_population.loc[
        solver_population["scale"] == s,
        "root_only"
    ].iloc[0]
) == 105

print(
    "✓ Supervisor-question analysis is internally consistent."
)

## 10. Reporting checklist

Use the outputs as follows:

- **Fig. 2:** `fig2_full_framework_three_stage.pdf`
- **Fig. 3:** `fig3_local_vs_full.pdf`
- **Fig. 4:** `fig4_delta_w_sensitivity.pdf`
- **Supervisor / Discussion:** `conflict_active_publication_summary.csv`, `runtime_breakdown.csv`, `solver_attenuation_summary.csv`
- **Optional explanatory figure:** `solver_attenuation_300x300.pdf`

Do not compare absolute runtime across the 50×50 machine and the Toubkal campaigns. Cross-scale interpretation should rely on ratios or within-scale comparisons, consistent with the experimental setup.